# ⚕️ Dual Diagnosis RAG — اجرای یک‌کلیکی در Google Colab

این نوت‌بوک:
1. کد را از GitHub می‌گیرد؛ 2. مقالات اعتبارسنجی‌شده را از Hugging Face دریافت می‌کند؛ 3. PDF/کتاب مجاز شما را می‌پذیرد؛ 4. ایندکس RAG را می‌سازد؛ 5. پرسش آزمایشی را اجرا می‌کند؛ 6. در صورت تنظیم کلید، نتیجه را در W&B ثبت می‌کند.

> **ایمنی:** ابزار آموزشی/کمک‌تصمیم است، جایگزین پزشک نیست. فقط فایل‌هایی را وارد کنید که اجازه استفاده از آن‌ها را دارید. اطلاعات هویتی بیمار را آپلود نکنید.


## 1) بررسی محیط و دریافت پروژه

In [ ]:
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB)
!rm -rf /content/dual-diagnosis-rag
!git clone -q https://github.com/SBZ-EDU/dual-diagnosis-rag.git /content/dual-diagnosis-rag
%cd /content/dual-diagnosis-rag
print('Repository ready')


## 2) نصب وابستگی‌های سبک RAG

In [ ]:
!pip -q install "sentence-transformers>=3,<4" "transformers>=4.44,<5" "huggingface_hub>=0.27,<2" "pypdf>=5,<6" "wandb>=0.18,<1" gradio
print('Dependencies installed')


## 3) دریافت دیتاست مقالات از Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil, json
src=hf_hub_download(repo_id='sosa123454321/dual-diagnosis-dataset',repo_type='dataset',filename='articles/validated_articles.jsonl')
dst=Path('data/articles/validated_articles.jsonl'); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy(src,dst)
rows=[json.loads(x) for x in dst.read_text(encoding='utf-8').splitlines() if x.strip()]
print(f'{len(rows)} validated article records downloaded')
for x in rows[:5]: print('•',x.get('title'),x.get('url'))


## 4) اختیاری: آپلود PDF مقاله/کتاب مجاز
PDFها به متن محلی تبدیل می‌شوند و فایل اصلی به Hugging Face یا W&B ارسال نمی‌شود مگر خودتان صریحاً این کار را انجام دهید.

In [ ]:
from pathlib import Path
if IN_COLAB:
    from google.colab import files
    uploaded=files.upload()  # اگر فایل ندارید این سلول را اجرا نکنید
    from pypdf import PdfReader
    out=Path('data/articles')
    for name,data in uploaded.items():
        if not name.lower().endswith('.pdf'): continue
        Path(name).write_bytes(data)
        reader=PdfReader(name)
        text='\n'.join((p.extract_text() or '') for p in reader.pages)
        safe=Path(name).stem.replace(' ','_')+'.md'
        (out/safe).write_text(f'# {Path(name).stem}\n\n{text}',encoding='utf-8')
        print(name,'->',out/safe,len(text),'characters')
else: print('Upload is available in Colab.')


## 5) ساخت ایندکس برداری RAG

In [ ]:
import os
os.environ['EMBED_MODEL']='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
!python -m scripts.build_index
from rag import store
chunks,vectors=store.load()
print('Chunks:',len(chunks),'Vector shape:',vectors.shape)


## 6) تست بازیابی مقاله

In [ ]:
from rag import retriever
question='در روان‌پریشی همراه مصرف مواد، در صورت لغزش چه ارزیابی‌هایی لازم است؟'
hits=retriever.search(question,top_k=5)
for i,h in enumerate(hits,1):
    print(f"\n[{i}] score={h['score']:.3f} source={h['source']}\n{h['text'][:700]}")


## 7) پاسخ RAG
حالت پیش‌فرض استخراجی و رایگان است. برای مدل مولد، GPU را از Runtime → Change runtime type فعال کنید.

In [ ]:
os.environ['USE_GENERATOR']='0'  # برای Colab رایگان سریع و قابل اتکا
from rag import pipeline
result=pipeline.answer(question)
print(result['answer'])
print('\nSources:')
for s in result['sources']: print('-',s)


## 8) ثبت ارزیابی در W&B (اختیاری)
کلید را فقط در Colab Secrets با نام `WANDB_API_KEY` ذخیره کنید؛ آن را داخل نوت‌بوک ننویسید.

In [ ]:
import os
try:
    if IN_COLAB:
        from google.colab import userdata
        os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY') or ''
    if os.getenv('WANDB_API_KEY'):
        import wandb
        run=wandb.init(project='dual-diagnosis-rag',entity='elasa2next-sosa-',job_type='colab-rag-rebuild',config={'chunks':len(chunks),'embedding_model':os.environ['EMBED_MODEL'],'generator':False})
        wandb.log({'chunks_indexed':len(chunks),'top_retrieval_score':float(hits[0]['score']) if hits else 0,'sources_returned':len(result['sources'])})
        run.finish(); print(run.url)
    else: print('WANDB_API_KEY not set; logging skipped safely.')
except Exception as e: print('W&B skipped:',e)


## 9) اجرای رابط Gradio در Colab (اختیاری)

In [ ]:
# اجرای این سلول یک لینک موقت Gradio می‌سازد.
# توجه: لینک با پایان نشست Colab خاموش می‌شود.
!USE_GENERATOR=0 GRADIO_SHARE=1 python app.py


## خروجی‌ها
- ایندکس: `index/chunks.json` و `index/vectors.npz`
- مقالات: `data/articles/`
- داشبورد: https://wandb.ai/elasa2next-sosa-/dual-diagnosis-rag
- مخزن مدل: https://huggingface.co/sosa123454321/dual-diagnosis-rag
- دیتاست: https://huggingface.co/datasets/sosa123454321/dual-diagnosis-dataset
